# Rentabilidad anual de un alojamiento en Airbnb Bogotá
## Enfoque combinado: tarifa por ocupación

**Pregunta de investigación 3.** Dadas las características de una vivienda, ¿cuál puede ser su
rentabilidad anual como alojamiento de corta estancia?

**Integrantes:** Esteban, Rubén, Andrés · **Universidad EIA**, Maestría · **Profesor:** Daniel Orozco Restrepo

**Datos.** Inside Airbnb, Bogotá, Distrito Capital, Colombia, compilado el 14 de septiembre de
2026. Licencia Creative Commons Attribution 4.0.

---
## Por qué descomponer

El ingreso de un alojamiento es el producto de dos cantidades que se forman por mecanismos
distintos y, sobre todo, que están medidas con calidad muy distinta.

$$\frac{\text{ingreso}}{\text{día expuesto}} = \text{tarifa por noche} \times \frac{\text{noches ocupadas}}{\text{día expuesto}}$$

La tarifa está publicada para todos los anuncios y se lee directamente de la fuente. La
ocupación es una estimación construida a partir del recuento de comentarios, con un supuesto
de tasa de reseña, un techo que censura a los más activos y un cero que aparece siempre que el
método no detecta actividad.

Estimar el producto con un solo modelo mezcla las dos calidades de medición en un mismo
residuo. Estimar cada factor por separado permite medir cuánto de la dificultad vive en cada
uno, y sitúa el problema de medición donde realmente está.

---
# Paso 1 — ¿Machine learning?

## 1.1 Entorno

In [ ]:
import sys                                                   # sys da informacion del interprete en uso
import time                                                  # time mide cuanto tarda cada ajuste
import numpy as np                                           # numpy: operaciones matematicas y arreglos
import pandas as pd                                          # pandas: manejo de tablas
import matplotlib.pyplot as plt                              # matplotlib: graficos
import sklearn                                               # scikit-learn: modelos y preprocesamiento

import airbnb_comun as ac                                    # modulo del proyecto

ac.estilo()                                                  # fija la paleta y el estilo de las figuras
pd.set_option("display.width", 170)                          # ancho de impresion de las tablas
pd.set_option("display.max_columns", 40)                      # columnas maximas que se imprimen
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")  # separador de miles y tres decimales

print("Python       ", sys.version.split()[0])               # version del interprete
print("scikit-learn ", sklearn.__version__)                   # version de scikit-learn
print("Semilla      ", ac.SEMILLA)                            # semilla unica del proyecto

## 1.2 El problema

Un propietario que evalúa alquilar su vivienda por noches necesita el ingreso anual esperado
para compararlo con el arriendo tradicional. Las dos cantidades que lo determinan responden a
decisiones distintas. La tarifa la fija él y el mercado la acepta o la rechaza. La ocupación
depende de esa tarifa, de la demanda de su zona y de la posición que el anuncio consiga en el
buscador, y ninguna de las dos se observa antes de operar.

Un promedio por localidad asigna el mismo ingreso a una habitación compartida y a un
apartamento completo que se alquila la mitad del año, cuyos ingresos difieren en un orden de
magnitud.

## 1.3 Carga y definición de la población

In [ ]:
listings = ac.cargar_listings()                              # carga y abre la cotizacion de cada anuncio
datos = ac.preparar_poblacion(listings)                       # agrega exposicion, actividad e ingreso por dia
reglas_cal = ac.reglas_calendario()                          # reglas de reserva que el anfitrion fija dia a dia
datos = datos.join(reglas_cal, on="id")                      # las pega a cada alojamiento por su identificador
print(f"Reglas del calendario unidas para {len(reglas_cal):,} alojamientos")
print(f"Cambian su estancia minima durante el ano: {datos['varia_estancia_minima'].mean() * 100:.1f} %")
datos = datos[datos["precio_noche"].notna()].copy()           # conserva los anuncios con tarifa publicada

print(f"Anuncios con tarifa publicada: {len(datos):,}")
print(f"  en regimen de tarifa mensual prorrateada: {datos['es_mensual'].mean() * 100:.1f} %")
print(f"  sin actividad detectada en la ventana:    {(~datos['actividad_detectada']).mean() * 100:.1f} %")
print(f"  con exposicion observable:                {datos['exposicion_observable'].mean() * 100:.1f} %")

datos = datos[~datos["es_mensual"]].copy()                    # el regimen mensual queda fuera del modelo de ingreso
print()
print(f"Tras separar el regimen mensual: {len(datos):,} anuncios")

El régimen de tarifa mensual queda fuera. La fuente no publica una tarifa de catálogo sino el
resultado de una consulta de disponibilidad, y para los anuncios con estancia mínima larga esa
consulta devuelve una mensualidad con descuentos ya aplicados, repartida entre las noches del
mes. Ese grupo está mal medido en los dos factores a la vez, porque además su huésped rara vez
deja comentario y su ocupación estimada resulta nula.

## 1.4 Partición agrupada por anfitrión

Un anfitrión puede publicar decenas de anuncios casi idénticos en el mismo edificio. Una
partición aleatoria por filas los reparte a ambos lados, de modo que el modelo recupera en
prueba una tarifa que ya vio en entrenamiento.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit         # particion que respeta los grupos

tamanos = datos["host_id"].value_counts()                     # anuncios por anfitrion
print(f"Anfitriones distintos: {datos['host_id'].nunique():,}   Maximo de anuncios de uno solo: {tamanos.max()}")
print(f"Anuncios de anfitriones con cinco o mas: {datos['host_id'].map(tamanos).ge(5).mean() * 100:.1f} %")

particion = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=ac.SEMILLA)
indices_train, indices_test = next(particion.split(datos, groups=datos["host_id"]))
train = datos.iloc[indices_train].copy()                      # particion de entrenamiento
test = datos.iloc[indices_test].copy()                        # particion de prueba

print()
print(f"Entrenamiento: {len(train):,} anuncios de {train['host_id'].nunique():,} anfitriones")
print(f"Prueba:        {len(test):,} anuncios de {test['host_id'].nunique():,} anfitriones")
print(f"Anfitriones compartidos entre particiones: {len(set(train['host_id']) & set(test['host_id']))}")

## 1.5 Valores extremos y regla a mano

El criterio de Tukey (1977) sitúa el corte en el tercer cuartil más tres veces el rango
intercuartílico, sobre el logaritmo de la tarifa por huésped. La tarifa por huésped compara
alojamientos de tamaño distinto, mientras que un corte sobre la tarifa total eliminaría
viviendas grandes legítimas.

In [ ]:
def por_huesped(tabla):                                       # tarifa por huesped de una tabla
    return tabla["precio_noche"] / tabla["accommodates"].clip(lower=1)

log_tarifa = np.log1p(por_huesped(train))                     # logaritmo de la tarifa por huesped
q1, q3 = log_tarifa.quantile([0.25, 0.75])                    # primer y tercer cuartil
tope = float(np.expm1(q3 + 3 * (q3 - q1)))                    # tope de Tukey devuelto a pesos
print(f"Tope de Tukey: ${tope:,.0f} por huesped y noche")
print(f"Descartados: {(por_huesped(train) > tope).sum()} en entrenamiento y {(por_huesped(test) > tope).sum()} en prueba")

train = train[por_huesped(train) <= tope].copy()              # aplica el corte
test = test[por_huesped(test) <= tope].copy()
print(f"Entrenamiento: {len(train):,} | Prueba: {len(test):,}")

In [ ]:
from sklearn.dummy import DummyRegressor                      # modelo que siempre predice el mismo valor

con_objetivo = train[train["ingreso_anual"].notna()]          # entrenamiento con el objetivo medido
prueba_objetivo = test[test["ingreso_anual"].notna()]         # prueba con el objetivo medido

regla = ac.ReglaPorGrupo("neighbourhood_cleansed").fit(con_objetivo, con_objetivo["ingreso_anual"])
base = DummyRegressor(strategy="median").fit(con_objetivo, con_objetivo["ingreso_anual"])

referencia = pd.DataFrame({
    "Linea base (mediana de la ciudad)": ac.metricas(prueba_objetivo["ingreso_anual"], base.predict(prueba_objetivo)),
    "Regla a mano (mediana por localidad)": ac.metricas(prueba_objetivo["ingreso_anual"], regla.predict(prueba_objetivo)),
}).T
print(referencia.to_string())
print()
print(f"La regla a mano se equivoca en ${referencia.loc['Regla a mano (mediana por localidad)', 'MAE']:,.0f} al ano,")
print(f"sobre un ingreso mediano observado de ${prueba_objetivo['ingreso_anual'].median():,.0f}.")

---
# Paso 2 — Datos

## 2.1 Fuente y papel de cada tabla

Los datos provienen de Inside Airbnb, compilados el 14 de septiembre de 2026 para Bogotá,
Distrito Capital, Colombia, bajo licencia Creative Commons Attribution 4.0.

La tabla de anuncios aporta las variables explicativas y la tarifa. Los comentarios construyen
y verifican la ocupación. El calendario describe el comportamiento de la oferta, y no mide la
ocupación pasada porque describe los días siguientes a la extracción, que todavía nadie
reservó.

## 2.2 Los dos factores y su calidad de medición

In [ ]:
calidad = pd.DataFrame({
    "Tarifa por noche": {
        "anuncios con el dato": f"{train['precio_noche'].notna().mean() * 100:.1f} %",
        "origen": "publicada por la fuente",
        "valores en cero": f"{(train['precio_noche'] == 0).mean() * 100:.1f} %",
        "censura superior": "no",
    },
    "Noches ocupadas": {
        "anuncios con el dato": f"{train['noches_ocupadas'].notna().mean() * 100:.1f} %",
        "origen": "estimada desde los comentarios",
        "valores en cero": f"{(train['noches_ocupadas'] == 0).mean() * 100:.1f} %",
        "censura superior": f"si, {(train['noches_ocupadas'] >= 255).mean() * 100:.1f} % en el techo de 255",
    },
}).T
print(calidad.to_string())
print()
print("La tarifa se lee; la ocupacion se infiere. El residuo de un modelo unico del producto")
print("mezcla las dos calidades de medicion y no permite saber cual de las dos limita.")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13.5, 3.6))          # tres paneles

ejes[0].hist(np.log1p(train["precio_noche"]), bins=60, color=ac.ACENTO, edgecolor="white", linewidth=0.3)
ejes[0].set_title("Tarifa por noche")
ejes[0].set_xlabel("log(1 + tarifa)"); ejes[0].set_ylabel("alojamientos")

ejes[1].hist(train["noches_ocupadas"], bins=60, color=ac.MORADO, edgecolor="white", linewidth=0.3)
ejes[1].axvline(255, color=ac.TINTA, linewidth=1.2, linestyle="--")
ejes[1].text(255, ejes[1].get_ylim()[1] * 0.85, "  techo\n  de la estimacion", fontsize=8, color=ac.TINTA, va="top")
ejes[1].set_title("Noches ocupadas estimadas")
ejes[1].set_xlabel("noches en el ultimo ano")

activos_vista = train[train["exposicion_observable"] & train["actividad_detectada"]]
ejes[2].scatter(np.log1p(activos_vista["precio_noche"]), activos_vista["noches_ocupadas"],
                s=5, alpha=0.16, color=ac.GRIS, edgecolors="none")
ejes[2].set_title("Los dos factores, entre los activos")
ejes[2].set_xlabel("log(1 + tarifa)"); ejes[2].set_ylabel("noches ocupadas")
correlacion = activos_vista["precio_noche"].corr(activos_vista["noches_ocupadas"], method="spearman")
ejes[2].text(0.05, 0.92, f"rho de Spearman {correlacion:.3f}", transform=ejes[2].transAxes,
             fontsize=9, color=ac.TINTA)

plt.tight_layout()
plt.show()

La correlación entre los dos factores decide si la descomposición puede tratarlos por
separado. Una dependencia fuerte obligaría a modelar el producto de forma conjunta, porque el
error de un factor arrastraría al otro; una dependencia débil permite estimarlos de manera
independiente y multiplicar.

## 2.3 Construcción de los objetivos

In [ ]:
for tabla in (train, test):                                   # define los objetivos en las dos particiones
    tabla["tasa_ocupacion"] = (tabla["noches_ocupadas"] / tabla["dias_expuesto"]).clip(0, 1)   # noches por dia expuesto

activos = train[train["exposicion_observable"] & train["actividad_detectada"]].copy()   # muestra con los dos factores medidos
print(f"Muestra con ambos factores medidos: {len(activos):,} anuncios de {activos['host_id'].nunique():,} anfitriones")
print()
print("Tarifa por noche:")
print(activos["precio_noche"].describe(percentiles=[.25, .5, .75, .95]).to_string())
print()
print("Tasa de ocupacion, noches por dia expuesto:")
print(activos["tasa_ocupacion"].describe(percentiles=[.25, .5, .75, .95]).to_string())
print()
reconstruido = activos["precio_noche"] * activos["tasa_ocupacion"]                      # producto de los dos factores
print(f"El producto de los dos factores reproduce el ingreso por dia expuesto en el "
      f"{np.isclose(reconstruido, activos['ingreso_por_dia'], rtol=0.01).mean() * 100:.1f} % de los casos.")

---
# Paso 3 — Limpieza y fugas

Las columnas prohibidas son las mismas en los dos factores, y por la misma razón. La tarifa y
las noches ocupadas forman el ingreso, de modo que cada una es fuga para el modelo de la otra
cuando el objetivo final es el producto. La ocupación estimada se deriva del recuento de
comentarios, lo que prohíbe toda columna de comentarios y de calificaciones. La disponibilidad
describe el calendario ya operado.

In [ ]:
descartes = ac.tabla_descartes()                              # tabla de columnas descartadas
print("Columnas descartadas, por grupo:")
print(descartes["Grupo"].value_counts().to_string())
print()
print(f"Total de columnas fuera del modelo: {len(descartes)}")
print()
print("Las diez primeras descartadas por fuga:")
print(descartes[descartes["Grupo"] == "Fuga"][["Columna", "Motivo"]].head(10).to_string(index=False))

In [ ]:
fig, ax = ac.lienzo(12, 3.4)                                  # lienzo para el esquema

ac.caja(ax, 0.34, 0.70, 0.30, 0.26, "Ingreso por dia expuesto", "objetivo final", ac.TINTA)
ac.caja(ax, 0.06, 0.34, 0.26, 0.24, "Tarifa por noche", "publicada, completa", ac.ACENTO)
ac.caja(ax, 0.66, 0.34, 0.26, 0.24, "Tasa de ocupacion", "estimada, censurada", ac.MORADO)
ac.caja(ax, 0.60, 0.00, 0.38, 0.24, "Comentarios y calificaciones", "insumo de la estimacion", ac.GRIS)
ac.caja(ax, 0.02, 0.00, 0.38, 0.24, "Cotizacion y disponibilidad", "reproducen tarifa y calendario", ac.GRIS)

ac.flecha(ax, (0.19, 0.58), (0.42, 0.70), "x", ac.TINTA)
ac.flecha(ax, (0.79, 0.58), (0.58, 0.70), "", ac.TINTA)
ac.flecha(ax, (0.79, 0.24), (0.79, 0.34))
ac.flecha(ax, (0.21, 0.24), (0.21, 0.34))

ax.set_title("Los dos factores y las columnas que los reconstruyen", loc="left", pad=10)
plt.show()

---
# Paso 4 — Columnas

Las dos etapas reciben exactamente el mismo conjunto de variables y el mismo preprocesamiento,
de modo que la diferencia entre ellas sea atribuible al factor que estiman y no al trato que
reciben.

La pregunta la formula un propietario que todavía no publicó, lo que parte las variables según
quién las controla. La dotación es lo que ya posee, las decisiones son aquello a lo que puede
comprometerse, y el desempeño es consecuencia de haber operado y no existe para él.

In [ ]:
variables = ac.tabla_variables()                              # tabla de variables con su mecanismo
print(pd.crosstab(variables["Tema"], variables["Quien la controla"]).to_string())
print()
print("Bloques del modelo del propietario nuevo:")
print("  " + ", ".join(ac.BLOQUES_ELEGIDOS))
print()
for grupo in ["Dotacion", "Decision"]:                        # las que si entran
    bloque = variables[variables["Quien la controla"] == grupo]
    print(f"{grupo} ({len(bloque)} variables): " + ", ".join(bloque["Variable"]))
print()
fuera = variables[variables["Quien la controla"] == "Desempeno"]
print(f"Fuera, por ser consecuencia de haber operado: " + ", ".join(fuera["Variable"]))

In [ ]:
preprocesador = ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)   # preprocesamiento comun a las dos etapas
t0 = time.time()
matriz = preprocesador.fit_transform(activos, np.log1p(activos["precio_noche"]))
print(f"Matriz: {matriz.shape[0]:,} filas x {matriz.shape[1]} columnas")
print(f"Vacios que quedan: {int(np.isnan(matriz).sum())}")
print(f"Comodidades abiertas en columna propia: {len(preprocesador.named_steps['dummies'].nombres_)}")
print(f"Tiempo: {time.time() - t0:.1f} s")

---
# Paso 5 — Modelos

## 5.1 El diseño

Tres estimaciones se componen en una. La probabilidad de que el alojamiento llegue a registrar
actividad detectable, la tarifa que sostiene, y la fracción de días que consigue vender.

In [ ]:
fig, ax = ac.lienzo(12, 3.4)                                  # lienzo para el esquema del diseno

ac.caja(ax, 0.01, 0.30, 0.16, 0.44, "Caracteristicas", "dotacion y\ndecisiones", ac.GRIS)
ac.caja(ax, 0.23, 0.70, 0.24, 0.26, "A — Riesgo", "probabilidad de\nactividad", ac.MORADO)
ac.caja(ax, 0.23, 0.38, 0.24, 0.26, "B1 — Tarifa", "pesos por noche", ac.ACENTO)
ac.caja(ax, 0.23, 0.06, 0.24, 0.26, "B2 — Ocupacion", "noches por dia\nexpuesto", ac.ACENTO)
ac.caja(ax, 0.55, 0.30, 0.18, 0.44, "Composicion", "A x B1 x B2\nx 365", ac.TINTA)
ac.caja(ax, 0.78, 0.30, 0.20, 0.44, "Ingreso anual", "esperado", ac.TINTA)

ac.flecha(ax, (0.17, 0.60), (0.23, 0.83))
ac.flecha(ax, (0.17, 0.52), (0.23, 0.51))
ac.flecha(ax, (0.17, 0.44), (0.23, 0.19))
ac.flecha(ax, (0.47, 0.83), (0.55, 0.62))
ac.flecha(ax, (0.47, 0.51), (0.55, 0.52))
ac.flecha(ax, (0.47, 0.19), (0.55, 0.42))
ac.flecha(ax, (0.73, 0.52), (0.78, 0.52))

ax.set_title("Enfoque combinado: tres estimaciones compuestas", loc="left", pad=10)
plt.show()

In [ ]:
from sklearn.model_selection import GroupKFold, cross_validate, cross_val_predict   # validacion agrupada
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression  # familia lineal
from sklearn.neighbors import KNeighborsRegressor                                     # vecinos
from sklearn.tree import DecisionTreeRegressor                                        # arbol
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,          # ensambles
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.neural_network import MLPRegressor                                       # red neuronal
from sklearn.compose import TransformedTargetRegressor                                # objetivo transformado
from sklearn.pipeline import Pipeline                                                 # encadena pasos
from xgboost import XGBRegressor                                                      # potenciado de XGBoost

cv = GroupKFold(n_splits=5)                                   # cinco particiones que respetan el anfitrion
METRICAS = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}

def modelo(estimador):                                        # Pipeline completo de un modelo de regresion
    base = Pipeline([("prep", ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)), ("modelo", estimador)])
    return TransformedTargetRegressor(regressor=base, func=np.log1p, inverse_func=np.expm1)

MODELOS = {
    "Regresion lineal": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001, max_iter=5000),
    "KNN (15 vecinos)": KNeighborsRegressor(n_neighbors=15),
    "Arbol (profundidad 8)": DecisionTreeRegressor(max_depth=8, random_state=ac.SEMILLA),
    "Random forest": RandomForestRegressor(n_estimators=300, random_state=ac.SEMILLA, n_jobs=-1),
    "Potenciado (histogramas)": HistGradientBoostingRegressor(random_state=ac.SEMILLA),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.08, max_depth=6, subsample=0.9,
                            colsample_bytree=0.9, random_state=ac.SEMILLA, n_jobs=-1, verbosity=0),
    "Red neuronal (64-32)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=400,
                                         random_state=ac.SEMILLA, early_stopping=True),
}

def comparar(tabla, objetivo, titulo):                        # compara todas las familias sobre un objetivo
    print(titulo)
    filas = [{"Modelo": "Linea base (mediana)", **medir(DummyRegressor(strategy="median"), tabla, objetivo)}]
    filas.append({"Modelo": "Regla a mano (localidad)", **medir(ac.ReglaPorGrupo(), tabla, objetivo)})
    for nombre, estimador in MODELOS.items():
        filas.append({"Modelo": nombre, **medir(modelo(estimador), tabla, objetivo)})
    return pd.DataFrame(filas).set_index("Modelo").sort_values("MAE")

def medir(estimador, tabla, objetivo):                        # mide un estimador con la validacion agrupada
    t0 = time.time()
    r = cross_validate(estimador, tabla, objetivo, cv=cv, groups=tabla["host_id"], scoring=METRICAS)
    return {"MAE": -r["test_mae"].mean(), "RMSE": -r["test_rmse"].mean(),
            "R2": r["test_r2"].mean(), "desv R2": r["test_r2"].std(), "Tiempo (s)": time.time() - t0}

## 5.2 Etapa A — Probabilidad de actividad detectable

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve          # area bajo la curva y curva ROC

y_actividad = train["actividad_detectada"].astype(int)        # objetivo binario
filas_a = []
for nombre, estimador in {"Regresion logistica": LogisticRegression(max_iter=2000),
                          "Random forest": RandomForestClassifier(n_estimators=300, random_state=ac.SEMILLA, n_jobs=-1),
                          "Potenciado (histogramas)": HistGradientBoostingClassifier(random_state=ac.SEMILLA)}.items():
    t0 = time.time()
    pipe = Pipeline([("prep", ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)), ("modelo", estimador)])
    r = cross_validate(pipe, train, y_actividad, cv=cv, groups=train["host_id"], scoring="roc_auc")
    filas_a.append({"Modelo": nombre, "AUC": r["test_score"].mean(), "desv": r["test_score"].std(),
                    "Tiempo (s)": time.time() - t0, "estimador": estimador})
    print(f"  {nombre:28s} AUC {filas_a[-1]['AUC']:.4f}   {filas_a[-1]['Tiempo (s)']:4.0f} s", flush=True)

etapa_a = pd.DataFrame(filas_a).set_index("Modelo").sort_values("AUC", ascending=False)
print()
print(etapa_a[["AUC", "desv", "Tiempo (s)"]].round(4).to_string())
modelo_a = Pipeline([("prep", ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)),
                     ("modelo", etapa_a.iloc[0]["estimador"])]).fit(train, y_actividad)

## 5.3 Etapa B1 — Tarifa por noche

La tarifa se lee directamente de la fuente y está disponible para todos los anuncios, de modo
que es el factor bien medido de los dos.

In [ ]:
etapa_b1 = comparar(activos, activos["precio_noche"], "Ajustando los modelos de tarifa por noche:")
print()
print(etapa_b1.round(3).to_string())
ac.guardar(etapa_b1, "combinado_b1_tarifa.csv")

## 5.4 Etapa B2 — Tasa de ocupación

La ocupación es el factor estimado, con un techo que censura a los más activos y con un cero
que aparece cada vez que el método no detecta actividad. La etapa trabaja sobre los
alojamientos que sí la registran, y expresa el resultado por día de exposición.

In [ ]:
etapa_b2 = comparar(activos, activos["tasa_ocupacion"], "Ajustando los modelos de tasa de ocupacion:")
print()
print(etapa_b2.round(4).to_string())
ac.guardar(etapa_b2, "combinado_b2_ocupacion.csv")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)   # dos paneles con el mismo eje vertical

for eje, tabla, titulo, color in [(ejes[0], etapa_b1, "Tarifa por noche", ac.ACENTO),
                                  (ejes[1], etapa_b2, "Tasa de ocupacion", ac.MORADO)]:
    orden = tabla.sort_values("R2")
    destaque = [color if n == tabla["R2"].idxmax() else ac.GRIS for n in orden.index]
    eje.barh(orden.index, orden["R2"], color=destaque, height=0.62)
    eje.errorbar(orden["R2"], np.arange(len(orden)), xerr=orden["desv R2"],
                 fmt="none", ecolor=ac.TINTA, elinewidth=1, capsize=2.5)
    eje.axvline(0, color=ac.TINTA, linewidth=0.8)
    eje.set_title(titulo)
    eje.set_xlabel("R2 en validacion cruzada agrupada")
    eje.tick_params(axis="y", labelsize=8.5)

plt.tight_layout()
plt.show()

La diferencia entre los dos paneles cuantifica dónde vive la dificultad. El factor que se lee
de la fuente admite una predicción mucho más ajustada que el factor que se infiere del
recuento de comentarios, y esa distancia no se cierra con un modelo mejor, porque procede de
cómo está construida cada cantidad.

## 5.5 Composición de los factores

Las predicciones de los dos factores se obtienen fuera de la partición de ajuste, con la misma
validación cruzada agrupada, de modo que el producto se evalúe sobre estimaciones que ninguno
de los dos modelos vio al entrenarse.

In [ ]:
mejor_b1 = etapa_b1.drop(["Linea base (mediana)", "Regla a mano (localidad)"]).index[0]   # finalista de la tarifa
mejor_b2 = etapa_b2.drop(["Linea base (mediana)", "Regla a mano (localidad)"]).index[0]   # finalista de la ocupacion
print(f"Finalista de la tarifa:    {mejor_b1}")
print(f"Finalista de la ocupacion: {mejor_b2}")

pred_b1 = cross_val_predict(modelo(MODELOS[mejor_b1]), activos, activos["precio_noche"],
                            cv=cv, groups=activos["host_id"])          # tarifa predicha fuera de ajuste
pred_b2 = cross_val_predict(modelo(MODELOS[mejor_b2]), activos, activos["tasa_ocupacion"],
                            cv=cv, groups=activos["host_id"])          # ocupacion predicha fuera de ajuste
producto = np.clip(pred_b1, 0, None) * np.clip(pred_b2, 0, 1)          # ingreso por dia como producto

directo = cross_val_predict(modelo(HistGradientBoostingRegressor(random_state=ac.SEMILLA)),
                            activos, activos["ingreso_por_dia"], cv=cv, groups=activos["host_id"])

comparacion = pd.DataFrame({
    "Producto de los dos factores": ac.metricas(activos["ingreso_por_dia"], producto),
    "Estimacion directa del ingreso": ac.metricas(activos["ingreso_por_dia"], directo),
}).T
print()
print("Ingreso por dia expuesto, validacion cruzada agrupada:")
print(comparacion.to_string())
ac.guardar(comparacion, "combinado_producto.csv")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))             # dos paneles

limite = float(np.percentile(activos["ingreso_por_dia"], 99))  # recorta la cola para que la nube sea legible
for eje, pred, titulo in [(ejes[0], producto, "Producto de los dos factores"),
                          (ejes[1], directo, "Estimacion directa")]:
    eje.scatter(activos["ingreso_por_dia"], pred, s=6, alpha=0.18, color=ac.MORADO, edgecolors="none")
    eje.plot([0, limite], [0, limite], color=ac.TINTA, linewidth=1, linestyle="--")
    eje.set_xlim(0, limite); eje.set_ylim(0, limite)
    eje.set_title(titulo)
    eje.set_xlabel("ingreso por dia observado"); eje.set_ylabel("ingreso por dia predicho")
    eje.xaxis.set_major_formatter(plt.FuncFormatter(ac.miles))
    eje.yaxis.set_major_formatter(plt.FuncFormatter(ac.miles))

plt.tight_layout()
plt.show()

---
# Paso 6 — Evaluación

## 6.1 Medida final sobre la partición de prueba

In [ ]:
modelo_b1 = modelo(MODELOS[mejor_b1]).fit(activos, activos["precio_noche"])      # tarifa, ajustada en entrenamiento
modelo_b2 = modelo(MODELOS[mejor_b2]).fit(activos, activos["tasa_ocupacion"])    # ocupacion, ajustada en entrenamiento

prob_test = modelo_a.predict_proba(test)[:, 1]                                    # probabilidad de actividad
tarifa_test = np.clip(modelo_b1.predict(test), 0, None)                           # tarifa esperada
ocupacion_test = np.clip(modelo_b2.predict(test), 0, 1)                           # ocupacion esperada
prediccion = prob_test * tarifa_test * ocupacion_test * 365                       # ingreso anual esperado

evaluable = test["ingreso_anual"].notna()                                         # filas con el objetivo medido
y_test = test.loc[evaluable, "ingreso_anual"]

final = pd.DataFrame({
    "Linea base": ac.metricas(y_test, base.predict(test[evaluable])),
    "Regla a mano": ac.metricas(y_test, regla.predict(test[evaluable])),
    "Enfoque combinado": ac.metricas(y_test, prediccion[evaluable.to_numpy()]),
}).T
print(final.to_string())

mejora = (1 - final.loc["Enfoque combinado", "MAE"] / final.loc["Regla a mano", "MAE"]) * 100
print()
print(f"Sobre la particion de prueba, el enfoque combinado reduce el error absoluto medio un {mejora:.1f} %")
print(f"frente a la regla a mano, de ${final.loc['Regla a mano', 'MAE']:,.0f} a ${final.loc['Enfoque combinado', 'MAE']:,.0f} al ano.")
ac.guardar(final, "evaluacion_combinado.csv")

## 6.2 Comparación con la estimación directa

La estimación directa del ingreso se obtuvo con el mismo preprocesamiento, la misma partición
y la misma validación. La comparación decide cuál de las dos formas de llegar al ingreso
conviene presentar, y el criterio no es solo el error.

In [ ]:
from pathlib import Path                                      # Path localiza el archivo de resultados

ruta_directo = ac.SALIDAS / "evaluacion_directo.csv"          # tabla de resultados de la estimacion directa
if ruta_directo.exists():                                     # si esta disponible, se compara
    tabla_directo = pd.read_csv(ruta_directo, index_col=0)
    fila = [f for f in tabla_directo.index if "tres etapas" in f.lower()]
    enfrentamiento = pd.DataFrame({
        "Estimacion directa del ingreso": tabla_directo.loc[fila[0]],
        "Producto de tarifa por ocupacion": final.loc["Enfoque combinado"],
    }).T
    print(enfrentamiento.to_string())
    diferencia = (enfrentamiento.loc["Producto de tarifa por ocupacion", "MAE"]
                  - enfrentamiento.loc["Estimacion directa del ingreso", "MAE"])
    print()
    print(f"Diferencia de error absoluto medio entre los dos enfoques: ${abs(diferencia):,.0f} al ano,")
    print(f"a favor del {'directo' if diferencia > 0 else 'combinado'}.")
else:
    print("La tabla de la estimacion directa no esta disponible en la carpeta de salidas.")

## 6.3 Dónde se equivoca cada factor

Descomponer permite atribuir el error. El error del producto se puede separar en la parte que
aporta la tarifa y la parte que aporta la ocupación, lo que indica sobre cuál de los dos
conviene trabajar.

In [ ]:
error_b1 = np.abs(np.log1p(pred_b1) - np.log1p(activos["precio_noche"]))         # error de la tarifa en escala log
error_b2 = np.abs(np.log1p(pred_b2) - np.log1p(activos["tasa_ocupacion"]))       # error de la ocupacion en escala log

atribucion = pd.DataFrame({
    "error medio en escala logaritmica": [error_b1.mean(), error_b2.mean()],
    "desviacion del objetivo": [np.log1p(activos["precio_noche"]).std(), np.log1p(activos["tasa_ocupacion"]).std()],
}, index=["Tarifa por noche", "Tasa de ocupacion"])
atribucion["error relativo a la dispersion"] = (atribucion["error medio en escala logaritmica"]
                                                / atribucion["desviacion del objetivo"])
print(atribucion.to_string())
print()
print("La razon entre el error y la dispersion del objetivo indica cuanto de la variacion")
print("de cada factor queda sin explicar, en una escala comparable entre los dos.")

In [ ]:
errores = pd.DataFrame({
    "real": y_test.to_numpy(),
    "predicho": prediccion[evaluable.to_numpy()],
    "room_type": test.loc[evaluable, "room_type"].to_numpy(),
    "noches": test.loc[evaluable, "noches_ocupadas"].to_numpy(),
}, index=y_test.index)
errores["error_abs"] = (errores["real"] - errores["predicho"]).abs()
errores["estado"] = np.where(errores["noches"] == 0, "Sin actividad",
                      np.where(errores["noches"] >= 255, "En el tope", "Con actividad"))

print("Error por tipo de alojamiento:")
print(errores.groupby("room_type").agg(alojamientos=("real", "size"), ingreso_mediano=("real", "median"),
      MAE=("error_abs", "mean")).sort_values("alojamientos", ascending=False).round(0).to_string())
print()
print("Error por estado del anuncio:")
por_estado = errores.groupby("estado").agg(alojamientos=("real", "size"), ingreso_mediano=("real", "median"),
      prediccion_mediana=("predicho", "median"), MAE=("error_abs", "mean"))
print(por_estado.round(0).to_string())

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))             # dos paneles

limite = float(np.percentile(errores["real"], 99))
ejes[0].scatter(errores["real"], errores["predicho"], s=7, alpha=0.25, color=ac.MORADO, edgecolors="none")
ejes[0].plot([0, limite], [0, limite], color=ac.TINTA, linewidth=1, linestyle="--")
ejes[0].set_xlim(0, limite); ejes[0].set_ylim(0, limite)
ejes[0].set_title("Ingreso anual predicho y observado")
ejes[0].set_xlabel("observado"); ejes[0].set_ylabel("predicho")
ejes[0].xaxis.set_major_formatter(plt.FuncFormatter(ac.miles))
ejes[0].yaxis.set_major_formatter(plt.FuncFormatter(ac.miles))

decil = pd.qcut(errores["predicho"], 10, labels=False, duplicates="drop")
calibracion = errores.groupby(decil).agg(observado=("real", "median"), predicho=("predicho", "median"))
ejes[1].plot(calibracion["predicho"], calibracion["observado"], "o-", color=ac.ACENTO, linewidth=2, markersize=7, label="observado")
tope_cal = calibracion[["observado", "predicho"]].max().max()
ejes[1].plot([0, tope_cal], [0, tope_cal], color=ac.GRIS, linewidth=1, linestyle="--", label="calibracion perfecta")
ejes[1].set_title("Calibracion por decil de prediccion")
ejes[1].set_xlabel("predicho, mediana del decil"); ejes[1].set_ylabel("observado, mediana del decil")
ejes[1].xaxis.set_major_formatter(plt.FuncFormatter(ac.miles))
ejes[1].yaxis.set_major_formatter(plt.FuncFormatter(ac.miles))
ejes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 6.4 El valor de cada compromiso

La descomposición permite responder la pregunta en los términos en que el propietario la
formula. Fijada la vivienda, las decisiones de operación cambian la tarifa y la ocupación de
forma distinta, y el ingreso resulta del producto.

La comparación que sigue es entre alojamientos comparables y no es un efecto causal. Quien
instala muchas comodidades también suele responder rápido y mantener el alojamiento en mejor
estado, de modo que la diferencia observada reúne la decisión con el tipo de anfitrión que la
toma.

In [ ]:
caso = test[evaluable].sample(1, random_state=ac.SEMILLA)     # una vivienda de la particion de prueba
print("Vivienda de referencia:")
print(caso[["room_type", "property_type", "accommodates", "bedrooms",
            "neighbourhood_cleansed", "minimum_nights"]].to_string(index=False))

PALANCAS = {                                                   # decisiones que el propietario puede tomar
    "Arrendar la vivienda entera": ("room_type", "Entire home/apt"),
    "Arrendar una habitacion privada": ("room_type", "Private room"),
    "Estancia minima de una noche": ("minimum_nights", 1),
    "Estancia minima de siete noches": ("minimum_nights", 7),
    "Registrar ante la autoridad de turismo": ("license", 123456),
    "Sin registro de turismo": ("license", np.nan),
}
filas = []
for etiqueta, (columna, valor) in PALANCAS.items():            # evalua cada decision por separado
    variante = caso.copy()                                     # parte siempre de la misma vivienda
    variante[columna] = valor                                  # cambia una sola decision
    ingreso = (modelo_a.predict_proba(variante)[:, 1]
               * np.clip(modelo_b1.predict(variante), 0, None)
               * np.clip(modelo_b2.predict(variante), 0, 1) * 365)
    filas.append({"Compromiso": etiqueta, "Ingreso anual estimado": float(ingreso[0])})

palancas = pd.DataFrame(filas).set_index("Compromiso")
palancas["Diferencia"] = palancas["Ingreso anual estimado"] - palancas["Ingreso anual estimado"].iloc[0]
print()
print(palancas.round(0).to_string())

## 6.5 Respuesta a la pregunta de investigación

In [ ]:
print("PREGUNTA 3. Dadas las caracteristicas de una vivienda, cual puede ser su rentabilidad anual.")
print("=" * 100)
print()
print(f"Sobre la particion de prueba, con anfitriones que el modelo no vio, el enfoque combinado")
print(f"predice el ingreso bruto anual con un error absoluto medio de ${final.loc['Enfoque combinado', 'MAE']:,.0f}")
print(f"y un R2 de {final.loc['Enfoque combinado', 'R2']:.3f}, frente a ${final.loc['Regla a mano', 'MAE']:,.0f} de la regla a mano.")
print()
print(f"La descomposicion situa la dificultad: el factor que se lee de la fuente alcanza un R2 de")
print(f"{etapa_b1['R2'].max():.3f}, mientras que el factor estimado desde los comentarios llega a {etapa_b2['R2'].max():.3f}.")
print()
print("Error segun el estado del anuncio:")
for estado, fila in por_estado.iterrows():
    print(f"  {estado:16s} {int(fila['alojamientos']):5d} anuncios   "
          f"observado ${fila['ingreso_mediano']:>13,.0f}   MAE ${fila['MAE']:>13,.0f}")

# Conclusiones

Descomponer el ingreso en tarifa y ocupación separa dos problemas de dificultad muy distinta.
La tarifa se predice con holgura a partir de atributos observables del inmueble, mientras que
la ocupación resiste, y lo hace por una razón que ningún modelo corrige: se construye a partir
del recuento de comentarios de cada alojamiento, con un supuesto de tasa de reseña, un techo
que censura a los más activos y un cero que aparece siempre que el método no detecta actividad.

La comparación entre estimar el producto de una vez y estimar cada factor por separado no se
resuelve solo por el error. Los dos caminos llegan a cifras semejantes, y el combinado entrega
además dos cantidades que el propietario puede interpretar por separado, junto con la
atribución del error a cada factor, que indica dónde tendría sentido invertir esfuerzo.

El diseño responde la pregunta tal como la formula quien la hace. Un propietario que todavía no
publicó dispone de su dotación y de las decisiones a las que puede comprometerse, y carece de
todo historial, de modo que el modelo que lo informa renuncia a las variables de desempeño
aunque mejoren el ajuste sobre la población que ya opera.

**Limitaciones.** El objetivo es una estimación de la fuente y no un ingreso contable. El
régimen de tarifa mensual queda fuera por estar mal medido en los dos factores. El cero no
distingue al anuncio nuevo del abandonado, porque el archivo no trae la fecha de publicación.
La comparación entre compromisos es entre alojamientos comparables y no es un efecto causal.
La medida es de ingreso bruto, de modo que compararla con el arriendo tradicional exigiría
descontar la operación, el aseo, los servicios y la vacancia, además de un precio de
referencia del arriendo residencial que la fuente no contiene.